# Exploration de Sellers — phase 1

Lecture seule, grain : une ligne par seller_id. Utiliser le .venv du projet sur C:.
Exécuter de haut en bas depuis la racine ou notebooks/. Aucun CSV corrigé ou exporté.
Les comparaisons avec Customers sont indicatives, sans référentiel géographique officiel.
Code exécuté avec Python le 2026-10-24 ; notebook livré sans sorties sauvegardées.
Rapport : docs/SELLERS_EXPLORATION.md.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import re
import sys
import unicodedata

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "AGENTS.md").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "AGENTS.md").is_file(), "Ouvrir depuis la racine ou notebooks/"
paths = {name: PROJECT_ROOT / "data" / filename for name, filename in {
    "sellers": "olist_sellers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
}.items()}
source_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest()
                 for name, path in paths.items()}
inferred = pd.read_csv(paths["sellers"])
sellers = pd.read_csv(paths["sellers"], dtype="string")
raw = pd.read_csv(paths["sellers"], dtype="string", keep_default_na=False)
items = pd.read_csv(paths["items"], usecols=["seller_id"], dtype="string")
customers = pd.read_csv(paths["customers"], usecols=[
    "customer_zip_code_prefix", "customer_city", "customer_state"], dtype="string")

def records(frame):
    return json.loads(frame.to_json(orient="records", force_ascii=False))

def counts(series):
    return {str(key): int(value) for key, value in series.value_counts().items()}

report = {
    "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                    "executable": sys.executable},
    "source_sha256": source_hashes,
    "source_size_bytes": paths["sellers"].stat().st_size,
    "shape": list(sellers.shape),
    "inferred_dtypes": {col: str(dtype) for col,dtype in inferred.dtypes.items()},
    "analysis_dtypes": {col: str(dtype) for col,dtype in sellers.dtypes.items()},
    "memory_deep_MiB": round(float(sellers.memory_usage(deep=True).sum())/2**20, 3),
    "missing": {col: int(value) for col,value in sellers.isna().sum().items()},
    "empty": {col: int(raw[col].eq("").sum()) for col in raw},
    "whitespace_only_nonempty": {col: int((raw[col].ne("") & raw[col].str.strip().eq("")).sum()) for col in raw},
    "outer_whitespace": {col: int(raw[col].ne(raw[col].str.strip()).sum()) for col in raw},
    "exact_duplicate_excess": int(sellers.duplicated().sum()),
    "duplicate_id_rows": int(sellers.seller_id.duplicated(keep=False).sum()),
    "unique_ids": int(sellers.seller_id.nunique()),
    "id_unusual_format": int((~sellers.seller_id.str.fullmatch("[0-9a-f]{32}", na=False)).sum()),
}
report

## Préfixes postaux et États

Un préfixe est un code, pas une mesure : une lecture entière perd ses zéros initiaux.
Le format proposé est cinq chiffres ASCII. La liste des États reprend la convention du validateur Customers existant.
Les distributions ne prouvent pas qu'un code postal est attribué officiellement.

In [ ]:
VALID_STATES = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT",
    "MS", "MG", "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO",
    "RR", "SC", "SP", "SE", "TO",
}
zip_codes = sellers.seller_zip_code_prefix
report["postal"] = {
    "length_distribution": counts(zip_codes.str.len()),
    "invalid_five_ascii_digits": int((~zip_codes.str.fullmatch("[0-9]{5}", na=False)).sum()),
    "leading_zero_rows": int(zip_codes.str.startswith("0", na=False).sum()),
    "all_zero_rows": int(zip_codes.eq("00000").sum()),
    "unique_prefixes": int(zip_codes.nunique()),
    "min_lexical": zip_codes.min(), "max_lexical": zip_codes.max(),
    "leading_zero_examples": zip_codes[zip_codes.str.startswith("0", na=False)].drop_duplicates().head(10).tolist(),
    "top_prefixes": counts(zip_codes) ,
}
report["states"] = {
    "distribution": counts(sellers.seller_state),
    "invalid_rows": int((~sellers.seller_state.isin(VALID_STATES)).sum()),
    "absent_states": sorted(VALID_STATES - set(sellers.seller_state.dropna())),
}
report["postal"], report["states"]

## Villes : variations et signaux géographiques

La clé exploratoire ci-dessous met en minuscules, retire les accents, remplace la ponctuation par des espaces et réduit les espaces.
Elle sert uniquement à repérer des variantes possibles ; aucune valeur source n'est remplacée.
Une ville dans plusieurs États peut être un homonyme, et un préfixe peut couvrir plusieurs libellés locaux.
Les cas observés sont des pistes d'examen, pas des corrections ni des erreurs confirmées.

In [ ]:
def city_key(value):
    if pd.isna(value):
        return None
    text = unicodedata.normalize("NFKD", value.casefold())
    text = "".join(char for char in text if not unicodedata.combining(char))
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", text)).strip()

geo = sellers.assign(city_key=sellers.seller_city.map(city_key))
variant_counts = geo.groupby(["seller_state", "city_key"]).seller_city.nunique()
variant_keys = variant_counts[variant_counts.gt(1)].index
variants = []
for state, key in variant_keys:
    group = geo[(geo.seller_state == state) & (geo.city_key == key)]
    variants.append({"state": state, "normalized_key": key, "rows": len(group),
                     "spellings": counts(group.seller_city)})
city_states = sellers.groupby("seller_city").seller_state.nunique()
prefix_states = sellers.groupby("seller_zip_code_prefix").seller_state.nunique()
prefix_cities = sellers.groupby("seller_zip_code_prefix").seller_city.nunique()
multi_state_prefixes = prefix_states[prefix_states.gt(1)].index
report["cities"] = {
    "unique_raw": int(sellers.seller_city.nunique()),
    "top20": dict(list(counts(sellers.seller_city).items())[:20]),
    "accented_rows": int(sellers.seller_city.map(lambda x: any(unicodedata.combining(c) for c in unicodedata.normalize("NFD", x))).sum()),
    "non_lowercase_rows": int(sellers.seller_city.ne(sellers.seller_city.str.lower()).sum()),
    "punctuation_rows": int(sellers.seller_city.str.contains(r"[^\w\s]", regex=True, na=False).sum()),
    "variant_groups": variants,
    "variant_group_rows": sum(group["rows"] for group in variants),
    "same_raw_city_multiple_states": {key: int(value) for key,value in city_states[city_states.gt(1)].items()},
}
report["geography_internal"] = {
    "prefixes_multiple_states": len(multi_state_prefixes),
    "multi_state_prefix_rows": records(sellers[sellers.seller_zip_code_prefix.isin(multi_state_prefixes)]),
    "prefixes_multiple_raw_cities": int(prefix_cities.gt(1).sum()),
    "multiple_city_prefix_examples": records(sellers[sellers.seller_zip_code_prefix.isin(prefix_cities[prefix_cities.gt(1)].index)].head(20)),
}
customer_geo = customers.rename(columns={
    "customer_zip_code_prefix": "prefix", "customer_city": "city", "customer_state": "state"})
customer_geo["city_key"] = customer_geo.city.map(city_key)
prefix_states_customer = customer_geo.groupby("prefix").state.agg(lambda values: set(values.dropna()))
prefix_cities_customer = customer_geo.groupby("prefix").city_key.agg(lambda values: set(values.dropna()))
known_prefix = zip_codes.isin(prefix_states_customer.index)
state_disagreement = sellers.apply(
    lambda row: row.seller_zip_code_prefix in prefix_states_customer.index
    and row.seller_state not in prefix_states_customer[row.seller_zip_code_prefix], axis=1)
city_disagreement = geo.apply(
    lambda row: row.seller_zip_code_prefix in prefix_cities_customer.index
    and row.city_key not in prefix_cities_customer[row.seller_zip_code_prefix], axis=1)
report["geography_customers_comparison"] = {
    "known_prefix_rows": int(known_prefix.sum()),
    "unseen_prefix_rows": int((~known_prefix).sum()),
    "state_not_observed_for_known_prefix_rows": int(state_disagreement.sum()),
    "city_key_not_observed_for_known_prefix_rows": int(city_disagreement.sum()),
    "state_disagreement_rows": records(sellers[state_disagreement]),
    "city_disagreement_examples": records(sellers[city_disagreement].head(20)),
}
report["cities"], report["geography_internal"], report["geography_customers_comparison"]

## Références Sellers / Order Items

Le contrôle d'appartenance distingue les lignes Item et les vendeurs distincts.
La jointure many_to_one sert à vérifier l'absence de multiplication : plusieurs articles par vendeur sont normaux.
Un vendeur sans article peut être légitime et reste un indicateur métier.

In [ ]:
unknown = ~items.seller_id.isin(sellers.seller_id)
unused = ~sellers.seller_id.isin(items.seller_id)
joined = items.merge(sellers[["seller_id", "seller_state"]], on="seller_id",
                     how="left", validate="many_to_one", indicator=True)
report["relationships"] = {
    "item_rows": len(items),
    "distinct_item_sellers": int(items.seller_id.nunique()),
    "missing_item_seller_ids": int((items.seller_id.isna() | items.seller_id.str.strip().eq("")).sum()),
    "unknown_seller_item_rows": int(unknown.sum()),
    "unknown_distinct_sellers": int(items.loc[unknown, "seller_id"].nunique()),
    "unused_seller_rows": int(unused.sum()),
    "joined_rows": len(joined),
    "items_per_seller_statistics": items.seller_id.value_counts().describe().to_dict(),
}
assert len(joined) == len(items)
assert source_hashes == {name: hashlib.sha256(path.read_bytes()).hexdigest()
                         for name,path in paths.items()}
report["relationships"]

## Synthèse et règles proposées — en attente de validation

Structure : quatre colonnes obligatoires ; seller_id non vide et unique ; préfixe postal renseigné de cinq chiffres ; ville et État non vides ; État dans la liste Customers.
Le format hexadécimal des identifiants est descriptif à ce stade.
Contrôle relationnel côté Order Items : seller_id doit exister dans Sellers, séparément des règles de Sellers.

Indicateurs : vendeurs non référencés, variantes de villes, préfixes multivilles/multi-États, écarts avec Customers.
Zéros initiaux et plusieurs vendeurs dans une ville sont normaux.
Aucune déduction automatique d'une ville/État correcte, aucune règle de rejet géographique non confirmée.
Rapport complet calculé ci-dessous ; aucun statut de qualité n'est attribué.

In [ ]:
print(json.dumps(report, ensure_ascii=False, indent=2))